# Unified Poisson Solver Testing
This notebook runs the unified test suites for Problem 1 (Table 1 & 2) as well as the complete NUDFT/NUFFT Accuracy Comparisons, tracking strictly $L_2$ relative errors and computational runtimes.

In [9]:
import os, sys
import pandas as pd
# Main project root
repo_root = r"C:\Users\charl\OneDrive\Documents\GitHub\NUFFTRR_Poisson"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from Tests.CPU.testing_helpers import (
    run_tests_pipeline,
    render_accuracy,
    render_runtime,
    render_table2_accuracy,
    render_table2_runtime
)


# Setup

In [10]:
N_vals_p0 = [32, 64, 128, 256, 512]
M_vals_p0 = [32, 64, 128, 256, 512]
N_fixed_p0 = 32

N_vals_p1 = [32, 64, 128, 256, 512]
M_vals_p1 = [32, 64, 128, 256, 512]
N_fixed_p1 = 32

N_vals_c = [32, 64, 128, 256, 512]
M_vals_c = [32, 64, 128, 256, 512]

MUTE_OUTPUT = True


# Run Problems

In [11]:
METHODS_P0 = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Fixed-jittered-NUDFT", label="Fixed jittered / NUDFT", azu_unif=1, mesh_kind="jittered", solver_azu_unif=1, use_nudft=True),
    dict(name="Fixed-jittered-NUFFT", label="Fixed jittered / NUFFT", azu_unif=1, mesh_kind="jittered", solver_azu_unif=1, use_nudft=False),
]

METHODS_P1 = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Fixed-rand-NUDFT", label="Fixed rand / NUDFT", azu_unif=1, mesh_kind="rand", solver_azu_unif=1, use_nudft=True),
    dict(name="Fixed-rand-NUFFT", label="Fixed rand / NUFFT", azu_unif=1, mesh_kind="rand", solver_azu_unif=1, use_nudft=False),
]

METHODS_COMP = [
    dict(name="Unif-FFT", label="Uniform / FFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=2, use_nudft=None),
    dict(name="Unif-NUDFT", label="Uniform / NUDFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=1, use_nudft=True),
    dict(name="Unif-NUFFT", label="Uniform / NUFFT", azu_unif=2, mesh_kind="uniform", solver_azu_unif=1, use_nudft=False),
]
for kind in ("rand", "jittered", "clustered", "sine"):
    METHODS_COMP += [
        dict(name=f"Fixed-{kind}-NUDFT", label=f"Fixed {kind} / NUDFT", azu_unif=1, mesh_kind=kind, solver_azu_unif=1, use_nudft=True),
        dict(name=f"Fixed-{kind}-NUFFT", label=f"Fixed {kind} / NUFFT", azu_unif=1, mesh_kind=kind, solver_azu_unif=1, use_nudft=False),
    ]

print("Running Problem 0 - Table 1 (Jittered)...")
df_p0_t1 = run_tests_pipeline(N_vals_p0, M_vals_p0, fixed_other=None, methods=METHODS_P0, test_type="P1_Table1",mute=MUTE_OUTPUT)
print("\nRunning Problem 0 - Table 2 (Jittered)...")
df_p0_t2 = run_tests_pipeline(None, M_vals_p0, fixed_other=N_fixed_p0, methods=METHODS_P0, test_type="P1_Table2",mute=MUTE_OUTPUT)

print("Running Problem 1 - Table 1 (Rand)...")
df_t1 = run_tests_pipeline(N_vals_p1, M_vals_p1, fixed_other=None, methods=METHODS_P1, test_type="P1_Table1",mute=MUTE_OUTPUT)
print("\nRunning Problem 1 - Table 2 (Rand)...")
df_t2 = run_tests_pipeline(None, M_vals_p1, fixed_other=N_fixed_p1, methods=METHODS_P1, test_type="P1_Table2",mute=MUTE_OUTPUT)


print("Running Accuracy Comparisons...")
df_cN = run_tests_pipeline(N_vals_c, None, fixed_other=32, methods=METHODS_COMP, test_type="Accuracy_VaryN", mute=MUTE_OUTPUT)
df_cM = run_tests_pipeline(None, M_vals_c, fixed_other=32, methods=METHODS_COMP, test_type="Accuracy_VaryM", mute=MUTE_OUTPUT)
print("Done.")

Running Problem 0 - Table 1 (Jittered)...

Running Problem 0 - Table 2 (Jittered)...
Running Problem 1 - Table 1 (Rand)...

Running Problem 1 - Table 2 (Rand)...
Running Accuracy Comparisons...
Done.


# Accuracy

In [15]:
print("--- ACCURACY SECTION ---")

print("Problem 0 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same")
render_accuracy(df_p0_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 0 - Table 1")
print("error along rows indicate nudft or nufft is not currently accurate enough")

print("Problem 0 - Table 2 - errors should decrease along rows")
render_table2_accuracy(df_p0_t2, title_prefix=f"Problem 0 - Table 2 (N={N_fixed_p0})")
print("Neumann error for nudft and nufft being constant is an issue. this is either with zero mode or handling fourier coeffs.")

print("Problem 1 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same")
render_accuracy(df_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 1 - Table 1")
print("error along rows indicate nudft or nufft is not currently accurate enough. Inaccuracy is worse than Problem 0, likely due to random mesh being more challenging than jittered.")

print("Problem 1 - Table 2 - errors should decrease along rows")
render_table2_accuracy(df_t2, title_prefix=f"Problem 1 - Table 2 (N={N_fixed_p1})")
print("for nonuniform case dirichlet and neumann errors being constant is concerning. Indicates either mode issue or accuracy issues with algorithm")

print("Problem 2 - Table 1 -All errors should be same along rows, and relatively same along columns")
render_accuracy(df_cN, index_col="N", columns_col="label", title_prefix="Accuracy Comparison (Fixed M=32)")
print("errors for diff meshes getting worse suggests nudft or nufft not accurate")

print("Problem 2 - Table 2 -All errors should decrease along rows, and relatively same along columns")
render_accuracy(df_cM, index_col="M", columns_col="label", title_prefix="Accuracy Comparison (Fixed N=32)")
print("errors for diff meshes getting worse suggests nudft or nufft not accurate")

--- ACCURACY SECTION ---
Problem 0 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same

Problem 0 - Table 1 Accuracy


error along rows indicate nudft or nufft is not currently accurate enough
Problem 0 - Table 2 - errors should decrease along rows

Problem 0 - Table 2 (N=32) Accuracy


Neumann error for nudft and nufft being constant is an issue. this is either with zero mode or handling fourier coeffs.
Problem 1 - Table 1 - errors should be the same for rows, and decrease along columns. uniform, nudft, and nufft should be same

Problem 1 - Table 1 Accuracy


error along rows indicate nudft or nufft is not currently accurate enough. Inaccuracy is worse than Problem 0, likely due to random mesh being more challenging than jittered.
Problem 1 - Table 2 - errors should decrease along rows

Problem 1 - Table 2 (N=32) Accuracy


for nonuniform case dirichlet and neumann errors being constant is concerning. Indicates either mode issue or accuracy issues with algorithm
Problem 2 - Table 1 -All errors should be same along rows, and relatively same along columns

Accuracy Comparison (Fixed M=32) Accuracy


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
N,,,,,,,,,,,
32,1.10e-05,1.10e-05,1.10e-05,1.49e-02,1.10e-05,1.10e-05,1.10e-05,1.91e-03,1.10e-05,1.06e-01,4.50e-03
64,1.10e-05,1.10e-05,1.10e-05,8.15e-02,1.10e-05,1.38e-02,1.11e-01,1.54e-03,1.10e-05,2.05e-01,1.40e-03
128,1.10e-05,1.10e-05,1.10e-05,2.34e-01,1.10e-05,1.68e+00,3.64e-02,1.72e-03,1.10e-05,2.79e-01,2.79e-04
256,1.10e-05,1.10e-05,1.10e-05,6.14e-01,1.10e-05,4.49e-01,4.67e-02,1.60e-03,1.10e-05,2.39e-01,1.49e-05
512,1.10e-05,1.10e-05,1.10e-05,1.86e-01,1.10e-05,9.97e-01,3.08e-02,1.21e-03,1.10e-05,2.32e-01,1.10e-05


errors for diff meshes getting worse suggests nudft or nufft not accurate
Problem 2 - Table 2 -All errors should decrease along rows, and relatively same along columns

Accuracy Comparison (Fixed N=32) Accuracy


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
M,,,,,,,,,,,
32,1.10e-05,1.10e-05,1.10e-05,1.49e-02,1.10e-05,1.10e-05,1.10e-05,1.91e-03,1.10e-05,1.06e-01,4.50e-03
64,2.66e-06,2.66e-06,2.66e-06,1.50e-02,2.66e-06,2.94e-06,2.66e-06,1.91e-03,2.66e-06,1.06e-01,4.50e-03
128,6.55e-07,6.55e-07,6.55e-07,1.50e-02,6.55e-07,1.52e-06,6.59e-07,1.91e-03,6.55e-07,1.06e-01,4.50e-03
256,1.62e-07,1.62e-07,1.62e-07,1.50e-02,1.62e-07,1.40e-06,1.72e-07,1.91e-03,1.63e-07,1.06e-01,4.50e-03
512,4.04e-08,4.04e-08,4.05e-08,1.49e-02,4.05e-08,1.45e-06,6.52e-08,1.91e-03,4.09e-08,1.06e-01,4.50e-03


errors for diff meshes getting worse suggests nudft or nufft not accurate


# Runtimes

In [13]:
print("--- RUNTIME SECTION ---")
render_runtime(df_p0_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 0 (Jittered) Table 1")
render_table2_runtime(df_p0_t2, title_prefix=f"Problem 0 (Jittered) Table 2 (N={N_fixed_p0})")

render_runtime(df_t1, index_col="N", columns_col=["label", "M"], title_prefix="Problem 1 (Rand) Table 1")
render_table2_runtime(df_t2, title_prefix=f"Problem 1 (Rand) Table 2 (N={N_fixed_p1})")

render_runtime(df_cN, index_col="N", columns_col="label", title_prefix="Accuracy Comparison (Fixed M=32)")
render_runtime(df_cM, index_col="M", columns_col="label", title_prefix="Accuracy Comparison (Fixed N=32)")

--- RUNTIME SECTION ---

Problem 0 (Jittered) Table 1 Runtime (s)



Problem 0 (Jittered) Table 2 (N=32) Runtime (s)



Problem 1 (Rand) Table 1 Runtime (s)



Problem 1 (Rand) Table 2 (N=32) Runtime (s)



Accuracy Comparison (Fixed M=32) Runtime (s)


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
N,,,,,,,,,,,
32,9.10e-04,0.0036,0.0240,0.0033,0.0026,0.0026,0.0024,0.4197,0.2323,0.6233,0.3315
64,9.07e-04,0.0045,0.0250,0.0038,0.0054,0.0042,0.0052,0.6483,0.3360,0.7264,0.5125
128,0.0011,0.0096,0.0270,0.0108,0.0156,0.0106,0.0103,0.7104,0.4926,0.6957,0.6244
256,0.0018,0.0218,0.0257,0.0243,0.0255,0.0259,0.0232,0.7484,0.5678,0.8267,0.6311
512,0.0029,0.2275,0.0286,0.3403,0.2947,0.2165,0.3420,0.8241,0.6234,0.8757,0.5059



Accuracy Comparison (Fixed N=32) Runtime (s)


label,Uniform / FFT,Uniform / NUDFT,Uniform / NUFFT,Fixed clustered / NUDFT,Fixed jittered / NUDFT,Fixed rand / NUDFT,Fixed sine / NUDFT,Fixed clustered / NUFFT,Fixed jittered / NUFFT,Fixed rand / NUFFT,Fixed sine / NUFFT
M,,,,,,,,,,,
32,0.0010,0.0042,0.0363,0.0043,0.0036,0.0034,0.0039,0.3405,0.2983,0.6417,0.3237
64,0.0012,0.0037,0.0397,0.0052,0.0046,0.0036,0.0043,0.7966,0.4576,1.0764,0.5122
128,0.0020,0.0079,0.0462,0.0071,0.0067,0.0072,0.0072,1.1028,0.6698,1.6905,0.7057
256,0.0038,0.0102,0.0782,0.0167,0.0189,0.0105,0.0108,2.1662,1.2060,3.2811,1.3368
512,0.0072,0.0494,0.1512,0.0489,0.0376,0.0341,0.0340,3.0185,2.2432,5.7903,2.6548
